# 03 描述性統計與 2×2 表 — 參考解答

松柏護理之家退伍軍人症群聚事件練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from epi_learning.metrics import risk_ratio

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

## 題目 1：COPD × 感染的 2×2 表

In [ ]:
# 建立 2×2 表
ct_copd = pd.crosstab(
    df["comorbidity_copd"], df["infected"],
    margins=True, margins_name="合計",
)
ct_copd.index = ["無 COPD", "有 COPD", "合計"]
ct_copd.columns = ["未感染", "感染", "合計"]
print(ct_copd)

# 提取四格
a = int(ct_copd.loc["有 COPD", "感染"])
b = int(ct_copd.loc["有 COPD", "未感染"])
c = int(ct_copd.loc["無 COPD", "感染"])
d = int(ct_copd.loc["無 COPD", "未感染"])

# RR
rr = risk_ratio(a, a + b, c, c + d)

# 95% CI
ln_rr = np.log(rr)
se = np.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
ci_lo = np.exp(ln_rr - 1.96 * se)
ci_hi = np.exp(ln_rr + 1.96 * se)

# 卡方檢定
chi2, p, _, _ = chi2_contingency([[a, b], [c, d]])

print(f"\nCOPD → 感染")
print(f"  RR = {rr:.3f} (95% CI: {ci_lo:.3f} – {ci_hi:.3f})")
print(f"  卡方 = {chi2:.3f}, p-value = {p:.4f}")

if ci_lo > 1:
    print("  → COPD 是感染的統計顯著危險因子")
else:
    print("  → COPD 與感染無統計顯著關聯（CI 包含 1）")

## 題目 2：各共病的 RR 排名

In [ ]:
comorbidities = [
    "comorbidity_chf", "comorbidity_dm",
    "comorbidity_cancer", "comorbidity_copd",
    "immunosuppressed",
]

results = []
for factor in comorbidities:
    ct = pd.crosstab(df[factor], df["infected"])
    a_i = int(ct.loc[1, 1])
    b_i = int(ct.loc[1, 0])
    c_i = int(ct.loc[0, 1])
    d_i = int(ct.loc[0, 0])
    rr_i = risk_ratio(a_i, a_i + b_i, c_i, c_i + d_i)
    chi2_i, p_i, _, _ = chi2_contingency([[a_i, b_i], [c_i, d_i]])
    ln_rr_i = np.log(rr_i)
    se_i = np.sqrt(1/a_i - 1/(a_i+b_i) + 1/c_i - 1/(c_i+d_i))
    ci_lo = np.exp(ln_rr_i - 1.96 * se_i)
    ci_hi = np.exp(ln_rr_i + 1.96 * se_i)
    results.append({
        "共病": factor.replace("comorbidity_", "").upper(),
        "RR": round(rr_i, 3),
        "95% CI": f"{ci_lo:.3f}–{ci_hi:.3f}",
        "p-value": round(p_i, 4),
        "顯著": "*" if ci_lo > 1 else "",
    })

rr_df = pd.DataFrame(results).sort_values("RR", ascending=False)
print("=== 各共病粗 RR 排名 ===")
print(rr_df.to_string(index=False))

## 題目 3（挑戰題）：性別差異分析

In [ ]:
# --- 性別 × 感染的 2×2 表 ---
# 先把 sex 轉成 0/1（M=1, F=0）方便建 crosstab
df["is_male"] = (df["sex"] == "M").astype(int)

ct_sex = pd.crosstab(df["is_male"], df["infected"])
a_s = int(ct_sex.loc[1, 1])  # 男+感染
b_s = int(ct_sex.loc[1, 0])  # 男+未感染
c_s = int(ct_sex.loc[0, 1])  # 女+感染
d_s = int(ct_sex.loc[0, 0])  # 女+未感染

rr_sex = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
chi2_s, p_s, _, _ = chi2_contingency([[a_s, b_s], [c_s, d_s]])

ln_rr_s = np.log(rr_sex)
se_s = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
ci_lo_s = np.exp(ln_rr_s - 1.96 * se_s)
ci_hi_s = np.exp(ln_rr_s + 1.96 * se_s)

print(f"男性 vs 女性 → 感染")
print(f"  RR = {rr_sex:.3f} (95% CI: {ci_lo_s:.3f} – {ci_hi_s:.3f})")
print(f"  p-value = {p_s:.4f}")

# --- 分性別 CFR ---
print(f"\n=== 分性別致死率 ===")
for sex_label in ["M", "F"]:
    infected_sex = df[(df["sex"] == sex_label) & (df["infected"] == 1)]
    deaths_sex = (infected_sex["outcome"] == "dead").sum()
    n_infected = len(infected_sex)
    cfr = deaths_sex / n_infected if n_infected > 0 else 0
    print(f"  {sex_label}: CFR = {cfr:.1%} ({deaths_sex}/{n_infected})")

### 解讀

- **感染風險（RR）**：如果 95% CI 包含 1，表示男女感染風險無顯著差異
- **致死率（CFR）**：即使感染風險相同，若 CFR 有差異，可能因為：
  - 男性共病較多（干擾）
  - 男性就醫延遲（行為因素）
  - 生理差異影響疾病嚴重度
- **結論**：感染風險（susceptibility）和預後（prognosis）是兩回事，需要分開分析